#### **Sistema de Recomendación E-Commerce — Filtrado Basado en Contenido (Content-Based Filtering)**
#### Equipo: MetricEdge
#### Autor: Deiberlyn Nin
---
##### **Contexto y Problema de Negocio**
En las plataformas de e-commerce, convertir la abundante información transaccional y de catálogo en sugerencias personalizadas es clave para mejorar la experiencia del cliente e incrementar la tasa de conversión. 

**Problema de negocio:** ¿Cómo sugerir a cada cliente los productos más afines a sus intereses y características particulares, asegurando que las recomendaciones mantengan alta relevancia comercial?

**Objetivo del Notebook:**  
Entrenar y evaluar un **Modelo de Filtrado Basado en Contenido**. Este enfoque calcula la afinidad y similitud entre el perfil del usuario (preferencias, historial y atributos) y las características textuales/categorías de los productos en el catálogo, generando como salida un ranking **Top-N de productos recomendados** por cliente.

##### **Enfoque Técnico en este Notebook**
* **Ingeniería de Características:** Extracción y vectorización de atributos de productos (categorías, rango de precio) y construcción del vector de preferencia del cliente.
* **Métrica de Similitud:** Similitud Coseno / Distancias de atributos entre vectores de cliente y producto.
* **Salida del Modelo:** Lista personalizada Top-N de productos con mayor score de afinidad.
* **Porcentaje de cobertura del catálogo de productos generados por el sistema de recomendación**
* **Métricas Técnicas de Evaluación:** Evaluado mediante *Precision@K*, *Recall@K* y *MAP@K* (Mean Average Precision).

##### **Fuentes de Datos Utilizadas**
**Fuente de origen (all dataset):** E-commerce Sales & Customer Analytics (150k), Kaggle
Para el desarrollo y entrenamiento de este modelo específico se integran dos tablas clave del pipeline ETL:
1. **product_clean.csv**: Contiene el catálogo formateado de productos, incluyendo identificadores (product_id), categorías (product_category), precios, márgenes y métricas aggregadas del producto.
2. **interacciones_clean.csv**: Registra la matriz de interacciones explícitas e implícitas entre los clientes y los productos (compras, frecuencia, montos y fechas) utilizada para construir las representaciones de perfil.

##### **Importación de Librerías**

In [3]:
# Librerias bases para manejos de tablas/operaciones matematicas
import pandas as pd
import numpy as np

# Extracción de dataset de forma dinamica
from pathlib import Path

# Librerias bases para graficas
import matplotlib.pyplot as plt
import seaborn as sns

# Para modulo o escalado de texto para el modelo
from sklearn.feature_extraction.text import TfidfVectorizer

# Para métricas de similitud
from sklearn.metrics.pairwise import cosine_similarity

# Para el preprocesamiento de datos y y diferentes escalados (variables numericas)
from sklearn.preprocessing import MinMaxScaler

# Libreria especial para manejo de matrices dispersas
import scipy.sparse as sp

# Para medir la Cobertura de predicción del modelo sobre el catalogo de los productos
from tqdm import tqdm

# Para extraer datos y que se pueda reproducir en otros pc (para el modelo hibrido)
import os
import joblib

#### **Carga de los Datasets**
Se realizó la lectura exitosa de los datasets preprocesados, confirmando las siguientes dimensiones para el modelado:

* **product_clean.csv**: 1.175 productos registrados en el catálogo.
* **interacciones_clean.csv**: 354.157 interacciones transaccionales registradas.

In [4]:
# 1. Carga/Extracción de los dataset previamente limpiados desde el pipeline ETL

BASE_DIR = Path.cwd().parent  # un nivel arriba de notebooks/, hasta la raiz del repo

interacciones_df = pd.read_csv(
    BASE_DIR / "data" / "processed" / "interacciones_clean.csv",
    parse_dates=["order_date"],
)
product_df = pd.read_csv(
    BASE_DIR / "data" / "processed" / "product_clean.csv"
)

print(f"Productos cargados: {product_df.shape[0]} filas")
print(f"Interacciones cargadas: {interacciones_df.shape[0]} filas")

Productos cargados: 1175 filas
Interacciones cargadas: 354157 filas


#### **Datos relevantes del Dataset**

##### Tabla: Catálogo de Productos (product_clean.csv)

| Columna | Definición |
| :--- | :--- |
| product_id | Identificador único del producto. |
| product_name | Nombre descriptivo del producto comercializado. |
| product_category | Categoría principal de clasificación del producto. |
| product_subcategory | Clasificación secundaria o subnivel de categoría. |
| brand | Marca comercial asociada al producto. |
| supplier | Proveedor o fabricante encargado del suministro. |
| unit_price | Precio de venta unitario al público. |
| product_cost | Costo directo de adquisición o producción por unidad. |
| product_rating | Calificación promedio asignada por los clientes (escala de 1 a 5). |

##### ***Importante:** Para nuestro modelo realizaremos un metadatos_text "combinando" las columnas *product_category, product_subcategory y brand*, para manejar nuestro texto no estructurado/palabras claves, asignandoles un peso numérico continuo a cada palabra según su relevancia. *Esta columna será la principal en alimentar el modelo. Explicación a detalle en el Punto 1.*

##### Tabla: Interacciones Cliente-Producto (interacciones_clean.csv)

| Columna | Definición |
| :--- | :--- |
| order_id | Código único de identificación de la orden de compra. |
| customer_id | Identificador único del cliente que realiza la transacción. |
| product_id | Identificador del producto adquirido en la orden. |
| order_date | Fecha de registro de la transacción. |
| order_status | Estado de la interacción (filtro con estados válidos: *Completed*, *Returned*). |
| quantity | Unidades compradas de un producto en la orden. |
| unit_price | Precio unitario al que se vendió el producto en la transacción. |
| discount_percentage | Porcentaje de descuento aplicado sobre la línea de venta. |
| discount_amount | Monto monetario total descontado. |
| gross_sales | Ingreso bruto generado antes de descuentos e impuestos. |
| tax_amount | Monto cobrado por concepto de impuestos. |
| shipping_cost | Costo de envío asociado a la línea de producto. |
| net_sales | Ingreso neto final percibido por la transacción. |
| product_cost | Costo de los productos vendidos en la interacción. |
| profit | Ganancia neta obtenida en la línea de la transacción. |

In [5]:
print("Datos y estructuras encontrados en la tabla Productos:")
display(product_df.head(3))

print("Datos y estructuras encontrados en la tabla interacciones:")
display(interacciones_df.head(3))

Datos y estructuras encontrados en la tabla Productos:


,product_id,product_name,product_category,product_subcategory,brand,supplier,unit_price,product_cost,product_rating
0,PROD-000001,Apple Sharable bifurcated algorithm Smartphones,Electronics,Smartphones,Apple,Alibaba,686.90,474.06,4.3
1,PROD-000002,Google User-centric even-keeled encryption Sma...,Electronics,Smartphones,Google,Euro Logistics,978.26,519.66,2.9
2,PROD-000003,LG Face-to-face client-driven support Smartphones,Electronics,Smartphones,LG,Asian Manufacturing,275.51,185.48,3.9


Datos y estructuras encontrados en la tabla interacciones:


,order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost,profit,customer_id,order_date,order_status
0,ORD-100004,PROD-000475,1,12.91,0.094500,1.22,12.91,2.10,10.10,23.89,4.26,9.53,CUST-022489,2025-03-29,Completed
1,ORD-100004,PROD-000782,1,116.11,0.098958,11.49,116.11,18.83,2.30,125.75,67.97,55.48,CUST-022489,2025-03-29,Completed
2,ORD-100004,PROD-000912,1,335.86,0.042012,14.11,335.86,57.92,4.06,383.73,195.75,183.92,CUST-022489,2025-03-29,Completed


#### **Punto 1. Construcción del modelo Filtrado Basado en Contenido (*Content-Based Filtering*)**
*Útilizando Matriz TF-IDF Vectorizer (Term Frequency - Inverse Document Frequency) | Matriz de Coseno de Similitud)*

##### **1. Proceso de Construcción del Modelo**

1. **Construcción de Metadatos Integrados:** Se combinaron las variables categóricas de texto (product_category, product_subcategory y brand) en la columna *metadata_text* para unificar los descriptores del producto.
2. **Vectorización Textual (TF-IDF):** Se aplicó *TfidfVectorizer* sobre la columna metadata_text para transformar el texto no estructurado en una matriz de frecuencias ponderadas, asignando pesos numéricos según la relevancia de cada palabra clave en el catálogo.
3. **Escalamiento de Atributos Continuos:** Se normalizaron las variables numéricas de precio (unit_price) y calificación (product_rating) en el rango *[0, 1]* mediante *MinMaxScaler*.
4. **Ensamblaje del Espacio Vectorial:** Se unieron la matriz dispersa de texto y las variables numéricas escaladas utilizando el formato comprimido **CSR** (*scipy.sparse.hstack*), optimizando el uso de memoria RAM.
5. **Cálculo de Similitud Coseno:** Se calculó la distancia angular entre todos los vectores de características para determinar la proximidad geométrica entre cada par de productos.

#### **2. Resultados de la Matriz de Representación**

* **Matriz de Atributos (item_features_matrix): *(1175, 321)***
  * **1,175 filas:** Representan la totalidad de productos únicos en el catálogo.
  * **321 columnas:** Conformadas por 319 términos descriptivos únicos extraídos por el vocabulario TF-IDF y 2 atributos numéricos escalados (precio y rating).
* **Matriz de Similitud Coseno (cosine_sim_matrix): *(1175, 1175)***
  * Matriz cuadrada de similitud entre pares, lista para realizar consultas de recomendación *O(1)* identificando los *K* ítems más cercanos en el espacio vectorial.

In [6]:
# Paso 1. Inicio de script para el Modelo de Filtrado Basado en Contenido, recomendando productos similares a otro producto.
# A través de la Matriz de Vectorización TF-IDF y Matriz de Similiud de Coseno.

# 1. Crear una metadata de texto combinando categoría, subcategoría y marca para la tabla product_df
product_model = product_df.copy()
product_model["metadata_text"] = (
    product_df["product_category"].fillna("")
    + " "
    + product_df["product_subcategory"].fillna("")
    + " "
    + product_df["brand"].fillna("")
)

# 2. Usaremos TF-IDF Vectorizer (Term Frequency - Inverse Document Frequency)
# Para nuestra nueva variable "metadata_text" en el dataset product_model, la cual será la principal de alimentar el modelo de recomendación.
# Ésta va a manejar nuestro texto no estructurado (palabras claves), asignando un peso numérico continuo a cada palabra según su relevancia.
# Convirtiendo en un vector de números cada variable diferente en la columna "metadata_text" (matriz TF-IDF).

"""
Resultado final buscado en este modelo: 
Si a un usuario le gusta un producto con esa metadata, el sistema le recomendará otros productos que tengan palabras clave similares 
en su metadata_text (por ejemplo, otros auriculares o productos de la misma marca).
"""

tfidf = TfidfVectorizer(stop_words=None)
tfidf_matrix = tfidf.fit_transform(product_model["metadata_text"])

# 3. Escalamiento de nuestras variables continuas/numéricas (Precio y Ranking)
scaler = MinMaxScaler()
numeric_features = scaler.fit_transform(
    product_model[["unit_price","product_rating"]].fillna(0)
)

# 4. Concatenamos nuestra matriz dispersa (TF-IDF) y nuestras features numericas
item_features_matrix = sp.hstack(
    [tfidf_matrix, numeric_features], format="csr" # CSR para compromir matrices con grandes cantidades de cero, ayuda a la memoria RAM
)

# 5. Cálculo de la Matriz de Similutud Coseno entre todos los productos (1175 x 1175)
cosine_sim_matrix = cosine_similarity(
    item_features_matrix, item_features_matrix
)

# 6. Visualizamos resultados a través de un print
print(f"Matriz de atributos generada: {item_features_matrix.shape}")
print(f"Matriz de Similitud de Coseno: {cosine_sim_matrix.shape}")

Matriz de atributos generada: (1175, 321)
Matriz de Similitud de Coseno: (1175, 1175)


#### **Punto 2. Validación del Motor de Recomendación Ítem a Ítem (Similitud Coseno)**

Se implementa y valida la función de búsqueda de ítems similares basada en la matriz de **similitud coseno** precalculada. El procedimiento sigue tres pasos clave:

1. **Mapeo de Índices:** Creación de una serie para mapear rápidamente cada product_id con su posición vectorial en la matriz.
2. **Función de Recomendación (product_recomendation):** Filtra y ordena de mayor a menor los puntajes de similitud entre un producto origen y el resto del catálogo, excluyendo el mismo producto.
3. **Prueba de Concepto:** Validación en tiempo real evaluando las recomendaciones generadas para el primer producto del catálogo.

#### Conclusión:
El motor de recomendación basado en contenido (*Item-to-Item Content-Based Filtering*) ya está completamente funcional.

Al consultar el producto **PROD-000001** (*Apple - Smartphones*), el modelo identifica correctamente ítems con alta cercanía en el espacio vectorial:

* **Top 1 (Puntaje *0.9851*):** PROD-000006 coincide en marca (Apple), categoría (Electronics) y subcategoría (Smartphones).
* **Top 2 (Puntaje *0.7838*):** PROD-000021 mantiene la marca (Apple) y la categoría (Electronics), pero varía la subcategoría (Laptops).
* **Top 3 y 4 (Puntajes *0.7577* y *0.7516*):** PROD-000004 y PROD-000012 mantienen la subcategoría (Smartphones) y categoría (Electronics), evaluando marcas competidoras (Sony y LG).

In [7]:
# Paso 1.1. Revisando resultados generados por matrices.
# Revisión de funcionalidad de matriz similitud de coseno creada previamente;
# Extrayendo desde product_id el primer producto y su relación con productos similares geometricamente

# 1. Mapeo de indice en product_id
index_product = pd.Series(
    product_model.index, index=product_model["product_id"]
).drop_duplicates()

# 2. Creación de función: Recomendación de productos según el indice ("product_id")
def product_recomendation (product_id, k_top=5):
    """
    Retorna los K productos más similares a un product_id dado usando Similitud Coseno.
    """
    if product_id not in index_product:
        return f"El producto {product_id} no existe en el catálogo."

    idx = index_product[product_id]

    # Puntaje de similitud del producto en comparación con todos los demás
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Ordenando de mayor a menor las similitudes, excluyendo el mismo producto
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1: k_top + 1]

    # Extrayendo los indices e información del catalogo
    product_index = [i[0] for i in sim_scores]

    resultado = product_model.iloc[product_index][
        [
            "product_id",
            "product_name",
            "product_category",
            "product_subcategory",
            "brand",
            "unit_price",
        ]
    ].copy()

    resultado["similarity_score"] = [
        round(score[1], 4) for score in sim_scores
    ]

    return resultado

# 3. Prueba de la función y matriz usando el primer producto del catalogo
sample_product, product_name = product_model[
    ["product_id", "product_name"]].iloc[0]
print(f"Recomendaciones para nuestro primer producto en el catálogo: {sample_product} - '{product_name}' ")
product_recomendation(sample_product, k_top=5)

Recomendaciones para nuestro primer producto en el catálogo: PROD-000001 - 'Apple Sharable bifurcated algorithm Smartphones' 


,product_id,product_name,product_category,product_subcategory,brand,unit_price,similarity_score
5,PROD-000006,Apple Optimized 5thgeneration algorithm Smartp...,Electronics,Smartphones,Apple,438.43,0.9851
20,PROD-000021,Apple Public-key bottom-line implementation La...,Electronics,Laptops,Apple,705.28,0.7838
3,PROD-000004,Sony Customer-focused systematic support Smart...,Electronics,Smartphones,Sony,1120.49,0.7577
11,PROD-000012,LG Reverse-engineered even-keeled workforce Sm...,Electronics,Smartphones,LG,1433.55,0.7516
31,PROD-000032,Apple Synergized secondary archive Headphones,Electronics,Headphones,Apple,929.66,0.7450


#### **Punto 3. Perfil de Usuario Basado en Contenido: Recomendación a un Cliente Específico**

En esta sección se construye un **vector de perfil del cliente** promediando las características de los productos pertenecientes a su historial transaccional previo. Posteriormente, se calcula la similitud entre este vector de preferencia y el catálogo general de productos para recomendar los ítems más afines.

El flujo de ejecución del script comprende:
1. **Generación del Vector de Perfil:** Extracción de los product_id comprados por el cliente CUST-022489 y promedio de sus representaciones vectoriales (item_features_matrix).
2. **Filtrado y Scoring:** Cálculo de la similitud coseno entre el perfil del usuario y el catálogo, descartando los ítems previamente adquiridos para evitar recomendaciones redundantes.
3. **Generación del Top-N:** Ordenamiento de los puntajes para retornar las *K* mejores sugerencias personalizadas.

#### **Conclusión e Insights de Negocio:**
* **Desempeño del Algoritmo:** El motor basado en contenido para usuarios está completamente funcional. Al analizar al cliente **CUST-022489**, cuyo historial muestra una fuerte recurrencia de compras en la categoría **Baby & Kids** (marcas como Huggies, Evenflo, Chicco y Pampers), el modelo recomendó 5 productos no comprados pertenecientes a esa misma categoría con scores de similitud sólidos (*0.7141* a *0.7388*).
* **Insight Técnico:** El modelo captura exitosamente el interés predominante del cliente en el catálogo actual. Sin embargo, al basarse exclusivamente en el promedio de atributos de compras pasadas, tiende a sobre-especializar las recomendaciones dentro de la categoría dominante.
* **Siguiente Paso / Mejora Recomendada:** Aunque este enfoque es ideal para clientes con historial consolidado, para maximizar la diversidad de la oferta (serendipidad) y evitar "burbujas de filtro", es recomendable hibridar este motor integrando **modelos de filtrado colaborativo** (Item-Based o Matrix Factorization) en las siguientes etapas del proyecto.

In [8]:
# Paso 2. Perfil de Usuario Basado en Contenido - Recomendación de productos a un usuario específico

interacciones_model = interacciones_df.copy()

# 1. Realizando función para extraer historial de compras por cliente
def customer_recomendation_product(customer_id, k_top=5):
    """"
    Genera recomendaciones basadas en contenido agregando el historial del cliente.
    """

    # Obteniendo compra de productos (historico) por clientes
    customer_products = interacciones_model[
        interacciones_model["customer_id"] == customer_id]["product_id"].unique()

    if len(customer_products) == 0:
        return f"El cliente {customer_id} no tiene historial de compras (Cold Start de Usuario)."

    # Indices de productos comprados en el catalogo

    index_customer_product = [
        index_product[pid]
        for pid in customer_products
        if pid in index_product
    ]

    if not index_customer_product:
        return f"Los productos del historial de {customer_id} no están en el catálogo."

    # Construyendo el vector de preferencias por usuario promediando los vectores de sus compras
    user_profile_vector = item_features_matrix[index_customer_product].mean(axis=0)

    # Arreglo a 2D para cosine_similarity
    user_profile_vector = np.asarray(user_profile_vector)

    # Calculando similitud entre: perfil del usuario y TODOS los productos del catalogo
    sim_scores = cosine_similarity(user_profile_vector, item_features_matrix)[0]

    # Ordenando productos por similitud y filtrando los que *YA* compro
    sim_scores_series = pd.Series(sim_scores, index=product_model["product_id"])
    sim_scores_filter = sim_scores_series.drop(
        labels = customer_products, errors="ignore"
    )

    top_product_ids = sim_scores_filter.nlargest(k_top).index

    # Configurando como mostrar los resultados
    resultado = product_model[
        product_model["product_id"].isin(top_product_ids)].copy()

    resultado["similarity_score"] = resultado["product_id"].map(
        sim_scores_filter
    )

    resultado = resultado.sort_values(
        by="similarity_score", ascending=False
    ).reset_index(drop=True)

    return resultado[
        [
            "product_id",
            "product_name",
            "product_category",
            "product_subcategory",
            "brand",
            "unit_price",
            "similarity_score",
        ]
    ]

# 2. Prueba utilizando el primer cliente en la tabla interacciones
# Escogiendo el primer cliente en la tabla interacciones
sample_customer = interacciones_model["customer_id"].iloc[0]

# 2.1 Generando el historial de compra de ese cliente
historial_pids = interacciones_model[
    interacciones_model["customer_id"] == sample_customer
]["product_id"].unique()

# 2.2 Buscar los nombres de los productos ya filtrados
historial_nombres = product_model[product_model["product_id"].isin(historial_pids)][
    ["product_id", "product_name", "product_category"]
]

# 2.3 Imprimiendo datos de compra del cliente (historial de compra)
print(f"Historial de compra del cliente: {sample_customer}")
print(f"Productos comprados previamente:")
display(historial_nombres)

# 2.4 Imprimiendo recomendación
print(f"RECOMENDACIONES GENERADAS PARA EL CLIENTE: {sample_customer}")

customer_recomendation_product(sample_customer, k_top=5)

Historial de compra del cliente: CUST-022489
Productos comprados previamente:


,product_id,product_name,product_category
12,PROD-000013,HP Adaptive well-modulated workforce Smartphones,Electronics
94,PROD-000095,Panasonic Re-engineered methodical encryption ...,Home Appliances
131,PROD-000132,Whirlpool Intuitive empowering database Coffee...,Home Appliances
201,PROD-000202,Nike Open-architected tangible frame Bags,Fashion
212,PROD-000213,Louis Vuitton Phased user-facing concept Watches,Fashion
271,PROD-000272,Benefit Versatile intangible extranet Fragrances,Beauty & Personal Care
314,PROD-000315,Wayfair Grass-roots next generation access Kit...,Home & Kitchen
337,PROD-000338,Home Depot Focused high-level conglomeration B...,Home & Kitchen
397,PROD-000398,Columbia Horizontal stable initiative Camping,Sports & Outdoors
458,PROD-000459,Wilson Inverse analyzing frame Swimming,Sports & Outdoors


RECOMENDACIONES GENERADAS PARA EL CLIENTE: CUST-022489


,product_id,product_name,product_category,product_subcategory,brand,unit_price,similarity_score
0,PROD-000913,Evenflo Profit-focused multimedia monitoring C...,Baby & Kids,Clothing,Evenflo,347.09,0.738801
1,PROD-000879,Huggies Fully-configurable cohesive Graphic In...,Baby & Kids,Diapers,Huggies,253.87,0.729548
2,PROD-000906,Chicco Automated client-driven structure Toys,Baby & Kids,Toys,Chicco,115.43,0.717689
3,PROD-000922,Pampers Focused uniform application Furniture,Baby & Kids,Furniture,Pampers,164.97,0.716355
4,PROD-000929,VTech Organic foreground instruction set Strol...,Baby & Kids,Strollers,VTech,301.02,0.714183


#### **Punto 4. Evaluación de la Cobertura del Catálogo (Catalog Coverage)**

En esta fase final se evalúa la **diversidad global del sistema** midiendo qué porcentaje del catálogo total es recomendado al generar las Top-5 sugerencias para una muestra representativa de *500* usuarios.

El proceso se ejecuta en tres pasos:
1. **Muestreo de Usuarios:** Selecciona los primeros *500* clientes con interacciones transaccionales en el dataset.
2. **Generación de Recomendaciones:** Ejecuta la función de recomendación (*K=5*) para cada cliente y acumula los identificadores de productos únicos recomendados.
3. **Cálculo del Metric:** Determina el porcentaje de ítems del catálogo que el modelo logra poner frente a los usuarios (*Catalog Coverage*).

#### **Conclusión y Resultados:**
* **Cobertura Alcanzada:** De los **1,175 productos** disponibles en el catálogo, el modelo recomendó **296 productos únicos** a la muestra de usuarios (500), alcanzando una cobertura del **25.19%**.
* **Eficiencia de Cómputo:** El pipeline demostró un alto rendimiento, procesando los 500 usuarios en aproximadamente 3.5 segundos (~139 iteraciones por segundo).
* **Interpretación de Negocio:** Una cobertura del *25.19\% con K=5* es esperable en un modelo de filtrado basado en contenido puramente determinista, ya que tiende a recomendar los ítems más centrales o representativos de cada categoría. 
* **Oportunidad de Mejora:** Para ampliar la visibilidad del *74.81\%* restante del catálogo (productos de larga cola o *long-tail*), se recomienda complementar este enfoque con modelos de **Filtrado Colaborativo** o aplicar penalizaciones por popularidad / filtros de diversidad.

In [9]:
# Punto 3. Final del modelo Basado en Contenido (Cobertura de predicción sobre el Catálogo)

# 1. Se selecciona una muestra representativa de usuarios
user_sample = interacciones_model["customer_id"].drop_duplicates().head(500)

recomendation_prods = set()
total_catalog_product = product_model["product_id"].nunique()

# 2. Recomendaciones (k_top=5) para cada usuario de la muestra (user_sample)
print("Calculando cobertura sobre una muestra de 500 usuarios...")
for customer_id in tqdm(user_sample):
    rec = customer_recomendation_product(customer_id, k_top=5)

    # Si retorna DataFrame con resultados, acumular los product_id
    if isinstance(rec, pd.DataFrame) and not rec.empty:
        recomendation_prods.update(rec["product_id"].tolist())

# 3. Porcentaje de cobertura
pct_cover = (len(recomendation_prods)/total_catalog_product) * 100

print("Resultados de Cobertura del Catálogo")
print(f"Productos únicos recomendados: {len(recomendation_prods)} de {total_catalog_product}")

print(f"\nCatalog Coverage (Top-5): {pct_cover:.2f}%")

Calculando cobertura sobre una muestra de 500 usuarios...


  1%|▏         | 7/500 [00:00<00:07, 67.80it/s]

100%|██████████| 500/500 [00:06<00:00, 79.13it/s]

Resultados de Cobertura del Catálogo
Productos únicos recomendados: 296 de 1175

Catalog Coverage (Top-5): 25.19%


#### **Validación Temporal y Evaluación Formal de Métricas de Ranking - Precision@K, Recall@K y MAP@K (SIN fuga de datos (data leakage))**

Para evaluar con rigor el rendimiento del modelo y evitar la fuga de información (*Data Leakage*), se implementó un esquema de validación temporal y cálculo de métricas de recomendación K-Top:

1. **División Temporal (Train/Test Split):** Se ordenaron las transacciones cronológicamente por fecha (order_date), asignando el *80\%* de los registros históricos para entrenamiento (train_df) y el *20\%* más reciente para prueba (test_df).
2. **Función de Recomendación Ajustada (evaluation_contend):** Construye el perfil del cliente utilizando **únicamente** su historial pasado en train_df, prediciendo sus Top-5 recomendaciones contra todo el catálogo.
3. **Métrica de Ranking (metrics_ranking):** Evalúa sobre una muestra aleatoria de usuarios las métricas de **Precision@5**, **Recall@5** y **MAP@5**, comparando los productos recomendados contra las compras reales futuras registradas en test_df (*Ground Truth*).

#### **Conclusión Final:**
El Modelo Basado en Contenido (Modelo B) registró en la evaluación formal un Precision@5 del 0.16%, un Recall@5 del 0.33% y un MAP@5 de 0.0009, junto con una cobertura del catálogo del 25.19%.
**Este resultado evidencia la limitación teórica del filtrado por contenido puro: es un enfoque excelente para resolver el Cold Start de producto y sugerir ítems semánticamente similares en la interfaz, pero no logra capturar por sí solo las tendencias complejas ni los cambios en la intención de compra de los usuarios a lo largo del tiempo.**

In [10]:
# 1. Ordenando las interacciones por fecha
interacciones_model = interacciones_model.sort_values(by="order_date").reset_index(drop=True)

# Definiendo el punto de corte al 80% del tiempo
split_idx = int(len(interacciones_model) * 0.8)

train_df = interacciones_model.iloc[:split_idx]
test_df = interacciones_model.iloc[split_idx:]

# Confirmando división exitosa
print(f"Interacciones de Entrenamiento (Train): {len(train_df)}")
print(f"Interacciones de Prueba (Test): {len(test_df)}")

# 2. Construyendo función de Recomendación Ajustada al Conjunto de Entrenamiento *Crucial para no generar Data Leakage*
def evaluation_contend(customer_id, df_train, k_top=5):

    # Se selecciona las compras del cliente en el conjunto de entrenamiento
    user_prods = df_train[df_train["customer_id"] == customer_id]["product_id"].unique()

    if len(user_prods) == 0:
        return []

    # Indices de la matriz de similitud
    index_customer_product = [index_product[pid] for pid in user_prods if pid in index_product]

    if not index_customer_product:
        return []

    # Vector de perfil promedio del cliente con sus compras pasadas
    user_profile = np.asarray(item_features_matrix[index_customer_product].mean(axis=0))
    
    # Similitud contra todo el catálogo
    sim_scores = cosine_similarity(user_profile, item_features_matrix)[0]
    
    sim_series = pd.Series(sim_scores, index=product_model["product_id"])
    sim_series = sim_series.drop(labels=user_prods, errors="ignore")
    
    # Devolver la lista con los IDs de los Top-K productos recomendados
    return sim_series.nlargest(k_top).index.tolist()

# 3. Creamos función para la evaluación de las Métricas seleccionando ambos conjuntos
def metrics_ranking(df_train, df_test, k_top=5, sample_size=500):

    # Usuarios de ambos conjuntos
    users_evaluation = sorted(
        list(set(df_train["customer_id"]).intersection(set(df_test["customer_id"])))
    )

    if len (users_evaluation) > sample_size:
        np.random.seed(42)
        users_evaluation = np.random.choice(users_evaluation, size=sample_size, replace=False)

    precisions = []
    recalls = []
    aps = []

    for customer_id in tqdm(users_evaluation, desc="Evaluando métricas de ranking"):

        ## Productos que el cliente COMPRÓ en el FUTURO (Ground Truth de Test)
        actual_future_purchases = set(df_test[df_test["customer_id"] == customer_id]["product_id"])
        
        # Productos que el modelo RECOMENDÓ en el Top-K usando solo Train
        recommended_items = evaluation_contend(customer_id, df_train, k_top=k_top)
        
        if not recommended_items or not actual_future_purchases:
            continue
        
        # Intersección: ¿Cuántos de los recomendados realmente los compró en Test?
        hits = set(recommended_items).intersection(actual_future_purchases)
        
        # Precision@K: aciertos / K
        precision = len(hits) / k_top
        
        # Recall@K: aciertos / total de compras reales futuras
        recall = len(hits) / len(actual_future_purchases)
        
        # Average Precision (AP@K) para el cálculo de MAP@K
        score = 0.0
        num_hits = 0.0
        for i, p in enumerate(recommended_items):
            if p in actual_future_purchases:
                num_hits += 1.0
                score += num_hits / (i + 1.0)
        ap = score / min(len(actual_future_purchases), k_top) if actual_future_purchases else 0.0
        
        precisions.append(precision)
        recalls.append(recall)
        aps.append(ap)
    
    mean_precision = np.mean(precisions)
    mean_recall = np.mean(recalls)
    map_score = np.mean(aps)

    # Visualizando resultados de las métricas (cuantos aciertos y desaciertos tuvimos)
    print(f"EVALUACIÓN FORMAL DEL MODELO (Top-{k_top})")
    print(f"Precision@{k_top}: {mean_precision:.4f} ({mean_precision * 100:.2f}%)")
    print(f"Recall@{k_top}:    {mean_recall:.4f} ({mean_recall * 100:.2f}%)")
    print(f"MAP@{k_top}:       {map_score:.4f}")
    
    return mean_precision, mean_recall, map_score

# Ejecución del cálculo
precision_5, recall_5, map_5 = metrics_ranking(train_df, test_df, k_top=5, sample_size=500)

Interacciones de Entrenamiento (Train): 283325
Interacciones de Prueba (Test): 70832


Evaluando métricas de ranking: 100%|██████████| 500/500 [00:10<00:00, 48.16it/s]

EVALUACIÓN FORMAL DEL MODELO (Top-5)
Precision@5: 0.0016 (0.16%)
Recall@5:    0.0032 (0.33%)
MAP@5:       0.0009


#### **Gráficas de Visualización de Resultados (Modelo)**

1. Rendimiento Real en Función de *K* en las métricas:
- **Recall@K (Línea Azul):** Crece de forma constante a medida que aumenta *K* (pasa de *0.0003* en *K=1* a más de *0.0080* en *K=15*). Es la tendencia matemática correcta: al ampliar la lista de recomendaciones, la probabilidad de capturar los productos comprados por el cliente en el período de prueba aumenta.
- **Precision@K (Línea Roja):** Mantiene una trayectoria plana cerca del *0.0020*. En catálogos grandes (*1,175* productos), recomendar más ítems incrementa ligeramente los aciertos totales pero diluye la proporción sobre el total *K*.

2. Cobertura del Modelo por Categoría de Producto
- **Distribución Homogénea:** Las barras verdes (productos recomendados en Top-5) se reparten entre todas las categorías del catálogo sin dejar ninguna en cero.
- **Categorías Destacadas:** Sports & Outdoors y Toys & Games registran los picos más altos de recomendación, mientras que categorías como Grocery tienen una menor presencia, alineándose directamente con los patrones reales de consumo presentes en el conjunto de entrenamiento.

3. Distribución de Similitud Coseno
- **Sesgo a la Izquierda Sano:** La mayor concentración de pares de productos se ubica entre *0.0* y *0.4*. Esto confirma que el espacio vectorial diferencia bien los ítems y no asigna puntuaciones de similitud altas de forma indiscriminada.
- **Filtro de Relevancia:** Solo una fracción pequeña del triángulo superior supera el score de *0.6*–*0.8*, garantizando que el algoritmo seleccione únicamente los ítems verdaderamente afines al construir el perfil del usuario.

In [11]:
# Gráfica 1. Cobertura por Categoria de Productos (cuántos productos únicos de esa categoría logró sugerir el modelo)
# Analizar productos recomendados en la muestra de cobertura
rec_products_df = product_model[
    product_df["product_id"].isin(recomendation_prods)
]

cat_total = product_model["product_category"].value_counts()
cat_rec = rec_products_df["product_category"].value_counts()

df_coverage = (
    pd.DataFrame({"Total Catálogo": cat_total, "Recomendados (Top-5)": cat_rec})
    .fillna(0)
    .reset_index()
)

# Gráfico de barras agrupadas
df_coverage.plot(
    x="product_category",
    kind="bar",
    figsize=(10, 5),
    color=["#a0c4ff", "#004b23"],
)
plt.title("Cobertura del Modelo por Categoría de Producto")
plt.xlabel("Categoría")
plt.ylabel("Número de Productos")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Gráfica 2. Distribución del Score de Similitud Coseno (para evidenciar que las similitudes esten bien distribuidas)
# Extraer el triángulo superior de la matriz de similitud (sin la diagonal)
upper_tri_indices = np.triu_indices_from(cosine_sim_matrix, k=1)
sim_values = cosine_sim_matrix[upper_tri_indices]

plt.figure(figsize=(8, 4))
sns.histplot(sim_values, bins=50, kde=True, color="#2b5c8f")
plt.title(
    "Distribución de Similitud Coseno entre Productos (Matriz 1175x1175)"
)
plt.xlabel("Score de Similitud Coseno")
plt.ylabel("Frecuencia (Pares de Productos)")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

# Gráfica 3. Curva de Precision@K vs Recall@K (Métricas de Ranking)
# 1. Definir los diferentes valores de K a evaluar
k_values = [1, 3, 5, 10, 15]

precision_scores = []
recall_scores = []

# 2. Iterar sobre cada K y calcular sus métricas reales
print("Calculando métricas para diferentes valores de K...")
for k in k_values:
    prec, rec, _ = metrics_ranking(
        train_df, test_df, k_top=k, sample_size=500
    )
    precision_scores.append(prec)
    recall_scores.append(rec)

# 3. Graficar los resultados reales obtenida por el modelo
plt.figure(figsize=(8, 4.5))

plt.plot(
    k_values,
    precision_scores,
    marker="o",
    linewidth=2,
    label="Precision@K",
    color="#d90429",
)
plt.plot(
    k_values,
    recall_scores,
    marker="s",
    linewidth=2,
    label="Recall@K",
    color="#118ab2",
)

plt.title(
    "Rendimiento Real del Modelo Basado en Contenido en Función de K",
    fontsize=12,
    pad=10,
)
plt.xlabel("Número de Recomendaciones (K)")
plt.ylabel("Score")
plt.xticks(k_values)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<Figure size 1000x500 with 1 Axes>

<Figure size 800x400 with 1 Axes>

Calculando métricas para diferentes valores de K...


Evaluando métricas de ranking: 100%|██████████| 500/500 [00:10<00:00, 48.83it/s]


EVALUACIÓN FORMAL DEL MODELO (Top-1)
Precision@1: 0.0000 (0.00%)
Recall@1:    0.0000 (0.00%)
MAP@1:       0.0000


...

Evaluando métricas de ranking: 100%|██████████| 500/500 [00:09<00:00, 50.48it/s]


EVALUACIÓN FORMAL DEL MODELO (Top-10)
Precision@10: 0.0024 (0.24%)
Recall@10:    0.0058 (0.58%)
MAP@10:       0.0012


Evaluando métricas de ranking: 100%|██████████| 500/500 [00:10<00:00, 48.90it/s]

EVALUACIÓN FORMAL DEL MODELO (Top-15)
Precision@15: 0.0033 (0.33%)
Recall@15:    0.0140 (1.40%)
MAP@15:       0.0018


<Figure size 800x450 with 1 Axes>

#### **Punto Final. Guardando Artefactos del Modelo (Necesario para el Modelo Hibrido y para trazabilidad)**

Para utilizar los componentes generados por este modelo en los notebooks de los otros integrantes o en el servicio de la API (FastAPI), los artefactos resultantes se han exportado y guardado en la carpeta ./models/:

##### Archivos Exportados
* **models/content_cosine_sim.joblib**: Matriz de similitud coseno precalculada entre los productos.
* **models/content_tfidf_vectorizer.joblib**: Objeto TfidfVectorizer entrenado con el vocabulario del catálogo.
* **models/content_scaler.joblib**: Escalador MinMaxScaler utilizado para normalizar las variables numéricas.

##### Instrucciones de Carga e Integración en Python

Para importar y utilizar estos artefactos desde otro notebook, script o desde el modelo híbrido, utiliza el siguiente bloque de código:

```python
import os
import joblib

# Ruta de destino
MODEL_DIR = "models"

# Carga de los artefactos exportados
cosine_sim_matrix = joblib.load(
    os.path.join(MODEL_DIR, "content_cosine_sim.joblib")
)
tfidf = joblib.load(
    os.path.join(MODEL_DIR, "content_tfidf_vectorizer.joblib")
)
scaler = joblib.load(
    os.path.join(MODEL_DIR, "content_scaler.joblib")
)

print(" Artefactos cargados exitosamente desde 'models/'")

```

In [12]:
# Carpeta models/ en la raiz del repo (no dentro de src/, siguiendo la estructura del proyecto)
MODELS_DIR = Path.cwd().parent / "models"

# Crear directorio de destino (si existe no hace nada)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Guardar matriz de similitud y transformadores
joblib.dump(
    cosine_sim_matrix, os.path.join(MODELS_DIR, "content_cosine_sim.joblib")
)
joblib.dump(tfidf, os.path.join(MODELS_DIR, "content_tfidf_vectorizer.joblib"))
joblib.dump(scaler, os.path.join(MODELS_DIR, "content_scaler.joblib"))

print("Artefactos exportados exitosamente en 'models/'")

Artefactos exportados exitosamente en 'models/'


##### **Revisando content_based.py se ejecute sin problemas.** 
*Conclusión: Se visualiza ejecución del pipeline del modelo, arroja predicciónes item-to-item, e item-to-user.*

In [13]:
import sys
sys.path.append('../src')
from content_based import ContentBasedEngine

engine = ContentBasedEngine()

# Prueba Item-to-Item
sample_item = product_model["product_id"].iloc[0]
print("-"*100)
print(f"Recomendaciones para producto: {sample_item}")
display(engine.product_recomendation(sample_item, k_top=3))

# Prueba User-to-Item
sample_user_purchases = train_df["product_id"].head(3).tolist()
print("-"*100)
print(f"Recomendaciones para historial de usuario")
display(
    engine.customer_recomendation_product(
        sample_user_purchases, k_top=3
    )
)

----------------------------------------------------------------------------------------------------
Recomendaciones para producto: PROD-000001


,product_id,product_name,product_category,product_subcategory,brand,supplier,unit_price,product_cost,product_rating,similarity_score
5,PROD-000006,Apple Optimized 5thgeneration algorithm Smartp...,Electronics,Smartphones,Apple,Domestic Producers,438.43,245.89,3.8,0.9851
20,PROD-000021,Apple Public-key bottom-line implementation La...,Electronics,Laptops,Apple,North American Supply,705.28,390.92,4.5,0.7838
3,PROD-000004,Sony Customer-focused systematic support Smart...,Electronics,Smartphones,Sony,Amazon Supply,1120.49,564.86,4.8,0.7577


----------------------------------------------------------------------------------------------------
Recomendaciones para historial de usuario


,product_id,product_name,product_category,product_subcategory,brand,supplier,unit_price,product_cost,product_rating,similarity_score
190,PROD-000191,Forever 21 Customizable human-resource orchest...,Fashion,Accessories,Forever 21,Global Trade,163.92,87.71,3.5,0.3773
178,PROD-000179,Forever 21 Integrated discrete definition Kids...,Fashion,Kids' Clothing,Forever 21,MultiSource,235.32,137.29,4.5,0.3302
186,PROD-000187,Forever 21 Fundamental 5thgeneration benchmark...,Fashion,Shoes,Forever 21,Alibaba,231.44,112.50,4.2,0.3293
